# Task Manager Eval Runner

**Following Teresa Torres' approach**: Simple notebook-based eval runner for systematic AI quality measurement.

This notebook provides:
1. **Data Loading**: Load traces and annotations from CSV files
2. **Eval Execution**: Run evals against traces with clear PASS/FAIL results
3. **Analysis Tools**: Compare human labels vs eval results
4. **Visualization**: Simple charts to track eval performance over time
5. **Debug Tools**: Investigate specific failures and edge cases

**Key Insight from Teresa**: "I know when I can measure something I can improve it."

## Setup and Imports

In [4]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from typing import Dict, List, Any

# Add evals to path
sys.path.append(os.path.join(os.getcwd(), 'evals'))

# Import our eval system
from evals.core import EvalResult, run_eval
from evals.trace_logging import TraceLogger, TaskTrace
from evals.annotation_tools import AnnotationManager, Annotation
from evals.task2_bdd_evals import (
    eval_no_requirement_invention,
    eval_implementation_contamination,
    eval_domain_consistency,
    eval_bdd_gold_standard_compliance
)
from evals.task3a_assessment_evals import (
    eval_automation_decisions,
    eval_assessment_criteria_adherence,
    eval_rationale_quality
)
from evals.cross_task_evals import (
    eval_p0_preservation,
    eval_traceability,
    eval_priority_consistency
)

print("✓ Imports successful")
print(f"Working directory: {os.getcwd()}")

ModuleNotFoundError: No module named 'pandas'

## Initialize Eval System

Create the core objects we'll use throughout the notebook.

In [ ]:
# Initialize managers
trace_logger = TraceLogger()
annotation_manager = AnnotationManager()

print("✓ Eval system initialized")
print(f"Traces directory: {trace_logger.log_dir}")
print(f"Annotations directory: {annotation_manager.annotation_dir}")

# Check what data we have
traces = trace_logger.load_traces()
annotations = annotation_manager.load_annotations()

print(f"\nData available:")
print(f"  Traces: {len(traces)}")
print(f"  Annotations: {len(annotations)}")

if traces:
    trace_summary = {}
    for trace in traces:
        if trace.task_name not in trace_summary:
            trace_summary[trace.task_name] = 0
        trace_summary[trace.task_name] += 1
    
    print(f"  Traces by task: {trace_summary}")

## Sample Data Creation

**If you don't have traces yet**, this creates sample traces using your existing CARCONF examples.

Skip this if you already have real trace data.

In [ ]:
# Create sample traces if we don't have any
if len(traces) == 0:
    print("No traces found. Creating sample data from CARCONF examples...")
    
    # Sample ticket content (from your examples)
    carconf_104_content = """
    📝 Jira Ticket: CARCONF-104
    Title: Paint Selection – User Interface Requirements
    
    Requirements:
    - User Paint Selection Interface
    - Given the user is on the paint selection page
    - When the user views available paint options for their selected model
    - Then the user should see all available paint colors displayed as visual swatches
    - And the user should see the paint name and additional cost for each option
    - And the user should be able to select one paint option
    - And the user should see their selection reflected in the configuration summary
    """
    
    carconf_106_content = """
    📝 Jira Ticket: CARCONF-106
    Title: Engine Selection – Technical Implementation Draft
    
    Requirements (Implementation-Focused):
    - Engine Selection API Integration
    - Given the frontend component mounts and calls GET /api/v2/engines?model=MODEL_ID
    - When the user clicks on div[data-engine-id] with onClick handler
    - Then the component should POST to /api/v2/configuration/selections with payload
    - And update the Redux store via dispatch(setSelectedEngine(engineId))
    - And render the EnginePreviewComponent with props.engineSpec
    """
    
    # Good BDD scenario (from CARCONF-104)
    good_bdd_scenario = """
    Feature: Paint Selection
    
    Scenario: User selects paint color
        Given user is on the paint selection page
        When user clicks on "Red Metallic" color swatch
        Then user sees paint selection confirmed in summary
        And user sees updated total price including paint cost
    """
    
    # Poor BDD scenario (contaminated)
    poor_bdd_scenario = """
    Feature: Engine Selection API
    
    Scenario: User selects engine via API
        Given the React component has mounted
        When user clicks the engine dropdown component
        Then the system should POST to /api/v2/engines endpoint
        And update Redux state with dispatch action
        And user should also see a engine comparison tool
        And system should provide recommendation engine
    """
    
    # Create sample traces
    sample_traces = [
        TaskTrace(
            trace_id="sample_task2_carconf104_good",
            task_name="task2_bdd",
            ticket_id="CARCONF-104",
            timestamp=datetime.now(),
            inputs={"ticket_content": carconf_104_content},
            outputs={"bdd_scenarios": good_bdd_scenario}
        ),
        TaskTrace(
            trace_id="sample_task2_carconf106_poor",
            task_name="task2_bdd",
            ticket_id="CARCONF-106",
            timestamp=datetime.now(),
            inputs={"ticket_content": carconf_106_content},
            outputs={"bdd_scenarios": poor_bdd_scenario}
        )
    ]
    
    # Save sample traces
    for trace in sample_traces:
        trace_logger.log_trace(trace)
    
    print(f"✓ Created {len(sample_traces)} sample traces")
    
    # Reload traces
    traces = trace_logger.load_traces()
    print(f"✓ Now have {len(traces)} total traces")
else:
    print(f"Using existing {len(traces)} traces")

## Eval Execution

**The core workflow**: Run all evals against all traces and collect results.

This is like Teresa's Sunday evening eval runs - systematic testing of AI quality.

In [ ]:
def run_all_evals_on_trace(trace: TaskTrace) -> Dict[str, EvalResult]:
    """Run all applicable evals on a single trace."""
    results = {}
    
    if trace.task_name == "task2_bdd":
        # Task 2 BDD Generation evals
        ticket_content = trace.inputs.get("ticket_content", "")
        bdd_scenarios = trace.outputs.get("bdd_scenarios", "")
        
        results["requirement_invention"] = run_eval(
            eval_no_requirement_invention, bdd_scenarios, ticket_content
        )
        
        results["implementation_contamination"] = run_eval(
            eval_implementation_contamination, bdd_scenarios
        )
        
        results["bdd_gold_standard"] = run_eval(
            eval_bdd_gold_standard_compliance, bdd_scenarios
        )
        
        # Cross-task evals
        results["p0_preservation"] = run_eval(
            eval_p0_preservation, ticket_content, bdd_scenarios, "task2_bdd"
        )
        
        results["traceability"] = run_eval(
            eval_traceability, bdd_scenarios, ticket_content
        )
    
    elif trace.task_name == "task3a_assessment":
        # Task 3a Assessment evals
        assessment_output = trace.outputs.get("assessment_result", "")
        scenarios = trace.inputs.get("bdd_scenarios", "")
        
        results["automation_decisions"] = run_eval(
            eval_automation_decisions, assessment_output, scenarios
        )
        
        results["criteria_adherence"] = run_eval(
            eval_assessment_criteria_adherence, assessment_output
        )
        
        results["rationale_quality"] = run_eval(
            eval_rationale_quality, assessment_output
        )
    
    return results

# Run evals on all traces
print("Running evals on all traces...")
all_eval_results = []

for i, trace in enumerate(traces):
    print(f"Processing trace {i+1}/{len(traces)}: {trace.trace_id}")
    
    eval_results = run_all_evals_on_trace(trace)
    
    # Store results with trace info
    for eval_name, result in eval_results.items():
        all_eval_results.append({
            "trace_id": trace.trace_id,
            "task_name": trace.task_name,
            "ticket_id": trace.ticket_id,
            "eval_name": eval_name,
            "status": result.status,
            "message": result.message,
            "timestamp": result.timestamp.isoformat()
        })

print(f"✓ Completed eval run: {len(all_eval_results)} results")

## Results Analysis

**Teresa's approach**: Convert results to a simple table for easy analysis.

"At a glance I can start to see how I'm doing."

In [2]:
# Convert results to DataFrame for analysis
results_df = pd.DataFrame(all_eval_results)

if len(results_df) > 0:
    print("Eval Results Summary:")
    print("=" * 50)
    
    # Overall pass/fail rates
    overall_stats = results_df['status'].value_counts()
    print(f"\nOverall Results:")
    for status, count in overall_stats.items():
        percentage = (count / len(results_df)) * 100
        print(f"  {status}: {count} ({percentage:.1f}%)")
    
    # Results by eval
    print(f"\nResults by Eval:")
    eval_stats = results_df.groupby(['eval_name', 'status']).size().unstack(fill_value=0)
    print(eval_stats)
    
    # Results by task
    print(f"\nResults by Task:")
    task_stats = results_df.groupby(['task_name', 'status']).size().unstack(fill_value=0)
    print(task_stats)
    
    # Display detailed results table (Teresa's approach)
    print(f"\nDetailed Results Table:")
    display_df = results_df.pivot_table(
        index=['trace_id', 'ticket_id'],
        columns='eval_name',
        values='status',
        aggfunc='first',
        fill_value='N/A'
    )
    
    print(display_df)
else:
    print("No eval results to display")

NameError: name 'pd' is not defined

## Failure Investigation

**Teresa's debug approach**: When evals fail, dig into the specific traces to understand why.

"I in my notebook I have tools that allow me to look into each of these failures."

In [ ]:
# Find and investigate failures
if len(results_df) > 0:
    failures = results_df[results_df['status'] == 'FAIL']
    
    if len(failures) > 0:
        print(f"Investigating {len(failures)} failures:")
        print("=" * 50)
        
        for _, failure in failures.iterrows():
            print(f"\n🔍 FAILURE INVESTIGATION")
            print(f"Trace: {failure['trace_id']}")
            print(f"Eval: {failure['eval_name']}")
            print(f"Message: {failure['message']}")
            
            # Find the original trace
            original_trace = next(
                (t for t in traces if t.trace_id == failure['trace_id']), None
            )
            
            if original_trace:
                print(f"\nTrace Details:")
                print(f"  Task: {original_trace.task_name}")
                print(f"  Ticket: {original_trace.ticket_id}")
                
                # Show relevant input/output
                if "bdd_scenarios" in original_trace.outputs:
                    scenarios = original_trace.outputs["bdd_scenarios"]
                    print(f"\n  Generated BDD Scenarios:")
                    print(f"  {'-' * 30}")
                    print(f"  {scenarios[:300]}...")  # First 300 chars
                
                if "ticket_content" in original_trace.inputs:
                    ticket = original_trace.inputs["ticket_content"]
                    print(f"\n  Original Ticket:")
                    print(f"  {'-' * 30}")
                    print(f"  {ticket[:200]}...")  # First 200 chars
            
            print(f"\n{'='*70}")
    else:
        print("🎉 No failures found! All evals passing.")
else:
    print("No results to investigate")

## Human vs Eval Comparison

**Teresa's key insight**: Compare human annotations with eval results to validate eval accuracy.

"This was a case where the code was better than I was."

In [ ]:
# Compare human annotations with eval results
if len(annotations) > 0 and len(results_df) > 0:
    print("Human vs Eval Comparison:")
    print("=" * 50)
    
    # Create comparison DataFrame
    comparison_data = []
    
    for annotation in annotations:
        # Find corresponding eval results
        eval_results = results_df[results_df['trace_id'] == annotation.trace_id]
        
        if len(eval_results) > 0:
            for _, eval_result in eval_results.iterrows():
                comparison_data.append({
                    'trace_id': annotation.trace_id,
                    'eval_name': eval_result['eval_name'],
                    'human_label': annotation.label,
                    'eval_label': eval_result['status'],
                    'match': annotation.label == eval_result['status'],
                    'human_confidence': annotation.confidence,
                    'failure_modes': ','.join(annotation.failure_modes)
                })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        
        # Overall alignment
        alignment_rate = comparison_df['match'].mean()
        print(f"Overall Alignment Rate: {alignment_rate:.2%}")
        
        # Alignment by eval
        print(f"\nAlignment by Eval:")
        eval_alignment = comparison_df.groupby('eval_name')['match'].agg(['mean', 'count'])
        eval_alignment['alignment_rate'] = eval_alignment['mean'].map(lambda x: f"{x:.2%}")
        print(eval_alignment[['alignment_rate', 'count']])
        
        # Show mismatches for investigation
        mismatches = comparison_df[~comparison_df['match']]
        if len(mismatches) > 0:
            print(f"\nMismatches to Investigate:")
            for _, mismatch in mismatches.iterrows():
                print(f"  Trace {mismatch['trace_id']} - {mismatch['eval_name']}:")
                print(f"    Human: {mismatch['human_label']} (confidence: {mismatch['human_confidence']})")
                print(f"    Eval: {mismatch['eval_label']}")
                if mismatch['failure_modes']:
                    print(f"    Human failure modes: {mismatch['failure_modes']}")
        
        print(f"\nComparison DataFrame:")
        display(comparison_df)
    else:
        print("No overlapping traces between annotations and eval results")
else:
    if len(annotations) == 0:
        print("No human annotations available for comparison")
        print("Run annotation tools to create human labels")
    if len(results_df) == 0:
        print("No eval results available for comparison")

## Visualization

**Teresa's approach**: Simple visualizations to track eval performance over time.

In [ ]:
# Create visualizations if we have data
if len(results_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Task Manager Eval Results Dashboard', fontsize=16)
    
    # Overall pass/fail distribution
    status_counts = results_df['status'].value_counts()
    axes[0, 0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%')
    axes[0, 0].set_title('Overall Pass/Fail Rate')
    
    # Results by eval type
    eval_pivot = results_df.pivot_table(
        index='eval_name', columns='status', values='trace_id', 
        aggfunc='count', fill_value=0
    )
    eval_pivot.plot(kind='bar', ax=axes[0, 1], color=['red', 'green'])
    axes[0, 1].set_title('Results by Eval Type')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Results by task
    task_pivot = results_df.pivot_table(
        index='task_name', columns='status', values='trace_id',
        aggfunc='count', fill_value=0
    )
    task_pivot.plot(kind='bar', ax=axes[1, 0], color=['red', 'green'])
    axes[1, 0].set_title('Results by Task')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Heatmap of results (if enough data)
    if len(results_df) > 5:
        heatmap_data = results_df.pivot_table(
            index='ticket_id', columns='eval_name', values='status',
            aggfunc=lambda x: 1 if 'PASS' in x.values else 0
        )
        sns.heatmap(heatmap_data, annot=True, cmap='RdYlGn', ax=axes[1, 1])
        axes[1, 1].set_title('Pass/Fail Heatmap by Ticket')
    else:
        axes[1, 1].text(0.5, 0.5, 'Need more data\nfor heatmap', 
                        ha='center', va='center', transform=axes[1, 1].transAxes)
        axes[1, 1].set_title('Pass/Fail Heatmap (Insufficient Data)')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\nEval Performance Summary:")
    print(f"{'='*40}")
    
    total_evals = len(results_df)
    passed_evals = len(results_df[results_df['status'] == 'PASS'])
    failed_evals = len(results_df[results_df['status'] == 'FAIL'])
    
    print(f"Total Evaluations: {total_evals}")
    print(f"Passed: {passed_evals} ({passed_evals/total_evals:.1%})")
    print(f"Failed: {failed_evals} ({failed_evals/total_evals:.1%})")
    
    # Top failure modes
    if failed_evals > 0:
        print(f"\nTop Failure Modes:")
        failure_modes = results_df[results_df['status'] == 'FAIL']['eval_name'].value_counts()
        for eval_name, count in failure_modes.head().items():
            print(f"  {eval_name}: {count} failures")
else:
    print("No results data available for visualization")

## Export and Next Steps

**Teresa's workflow**: Export results for further analysis and planning improvements.

In [ ]:
# Export results for further analysis
if len(results_df) > 0:
    # Create exports directory
    export_dir = os.path.join(os.getcwd(), 'eval_exports')
    os.makedirs(export_dir, exist_ok=True)
    
    # Export detailed results
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    results_file = os.path.join(export_dir, f'eval_results_{timestamp}.csv')
    results_df.to_csv(results_file, index=False)
    print(f"✓ Exported eval results to: {results_file}")
    
    # Export summary statistics
    summary_file = os.path.join(export_dir, f'eval_summary_{timestamp}.txt')
    with open(summary_file, 'w') as f:
        f.write("Task Manager Eval Run Summary\n")
        f.write(f"Generated: {datetime.now().isoformat()}\n")
        f.write(f"Total Evaluations: {len(results_df)}\n")
        f.write(f"Pass Rate: {(results_df['status'] == 'PASS').mean():.2%}\n")
        f.write("\nResults by Eval:\n")
        f.write(str(results_df.groupby(['eval_name', 'status']).size().unstack(fill_value=0)))
    
    print(f"✓ Exported summary to: {summary_file}")
    
    # Next steps recommendations
    print(f"\n🎯 Next Steps:")
    
    failed_evals = results_df[results_df['status'] == 'FAIL']['eval_name'].value_counts()
    if len(failed_evals) > 0:
        worst_eval = failed_evals.index[0]
        print(f"1. Focus on fixing '{worst_eval}' eval (most failures)")
        print(f"2. Investigate failure patterns in the traces")
        print(f"3. Consider if eval is too strict or AI needs improvement")
    else:
        print(f"1. 🎉 All evals passing! Consider adding more challenging test cases")
        print(f"2. Add more traces to increase eval coverage")
    
    print(f"4. Collect human annotations for eval validation")
    print(f"5. Run evals after each AI model/prompt change")
    
else:
    print("No results to export. Create traces first by running your task manager system.")

## Quick Eval Functions

**Convenience functions** for running evals on new data without going through the full notebook.

In [ ]:
def quick_eval_bdd_scenario(ticket_content: str, bdd_scenario: str, ticket_id: str = "manual_test"):
    """Quick evaluation of a single BDD scenario."""
    print(f"Quick Eval: {ticket_id}")
    print(f"{'='*50}")
    
    # Run key BDD evals
    results = {
        "requirement_invention": run_eval(eval_no_requirement_invention, bdd_scenario, ticket_content),
        "implementation_contamination": run_eval(eval_implementation_contamination, bdd_scenario),
        "bdd_gold_standard": run_eval(eval_bdd_gold_standard_compliance, bdd_scenario),
        "traceability": run_eval(eval_traceability, bdd_scenario, ticket_content)
    }
    
    # Display results
    for eval_name, result in results.items():
        status_icon = "✅" if result.passed else "❌"
        print(f"{status_icon} {eval_name}: {result.status}")
        if result.failed:
            print(f"   └─ {result.message}")
    
    return results

def eval_health_check():
    """Check if eval system is working properly."""
    print("Eval System Health Check")
    print(f"{'='*30}")
    
    # Test with known good/bad examples
    good_scenario = "Given user is on page\nWhen user clicks button\nThen user sees result"
    bad_scenario = "Given React component mounts\nWhen API POST request\nThen Redux store updates"
    
    # Test implementation contamination eval
    good_result = run_eval(eval_implementation_contamination, good_scenario)
    bad_result = run_eval(eval_implementation_contamination, bad_scenario)
    
    if good_result.passed and bad_result.failed:
        print("✅ Implementation contamination eval working correctly")
    else:
        print("❌ Implementation contamination eval not working properly")
        print(f"   Good scenario result: {good_result.status}")
        print(f"   Bad scenario result: {bad_result.status}")
    
    print(f"✅ Eval system health check complete")

# Run health check
eval_health_check()

## Usage Examples

**How to use this notebook** in your daily workflow.

In [ ]:
# Example: Test a new BDD scenario
print("Example: Testing a new BDD scenario")
print("=" * 40)

example_ticket = """
User Story: As a customer, I want to select a paint color 
so that I can customize my car's appearance.

Acceptance Criteria:
- User can view available paint colors
- User can select one paint color
- User sees selection confirmed
"""

example_scenario = """
Feature: Paint Selection

Scenario: User selects paint color
    Given user is on the paint selection page
    When user clicks on "Blue Metallic" paint swatch
    Then user sees "Blue Metallic" selected in summary
    And user sees updated price with paint cost
    And user should also see paint comparison tool
"""

# Run quick eval
results = quick_eval_bdd_scenario(example_ticket, example_scenario, "EXAMPLE-001")

print(f"\nThis scenario would FAIL because:")
print(f"- Contains scope expansion: 'should also see paint comparison tool'")
print(f"- This adds functionality not mentioned in the ticket")
print(f"\nThis demonstrates how evals catch AI 'creativity' that adds unasked features.")

---

## Summary

**You now have a complete eval system** following Teresa Torres' approach:

✅ **Simple infrastructure** - CSV-based trace logging and annotation tools  
✅ **Domain-focused evals** - Catches requirement invention, implementation contamination  
✅ **Fast feedback loops** - Run evals after every AI change  
✅ **Human validation** - Compare eval results with human annotations  
✅ **Debug tools** - Investigate specific failures  
✅ **Progress tracking** - Visualize improvements over time  

**Next steps**:
1. Integrate trace logging into your task manager execution
2. Run this notebook after each AI model/prompt change
3. Annotate failures to validate eval accuracy
4. Use eval failures to guide AI improvements

**Key insight from Teresa**: "I know when I can measure something I can improve it."

Your task manager system can now systematically improve AI quality instead of hoping it works.